# Feature Separation and Recalibration (FSR) — CVPR 2023
**Lightning AI Version** — adapted for Lightning AI Studios with H200 GPU.

---
## ⚠️ BEFORE RUNNING — Select H200 GPU

1. Open your **Lightning AI Studio**
2. Click the **compute icon** (top-right, looks like a chip)
3. Select **H200 GPU** from the list
4. Wait ~30 seconds for the machine to switch
5. Then run all cells **top to bottom in order**

---
## Key differences vs Kaggle / Colab

| | Colab | Kaggle | Lightning AI |
|---|---|---|---|
| Working directory | `/content/FSR` | `/kaggle/working/FSR` | `/teamspace/studios/this_studio/FSR` |
| Save weights | Google Drive | `/kaggle/working/` | `/teamspace/studios/this_studio/` |
| Session length | ~12 hrs | 9 hrs | **4 free hrs / month on H200** |
| Storage persists? | ❌ Resets | ⚠️ Partial | ✅ **Fully persistent** |
| Internet | On | Must enable | ✅ On by default |

> **Lightning AI storage is fully persistent.** Everything saved to
> `/teamspace/studios/this_studio/` stays there across sessions.
> If your session ends mid-training, just reopen the Studio and run
> Cell 7 again — it resumes from the last saved epoch automatically.
>
> ⚡ **H200 (80 GB VRAM) is ~5–6× faster than T4 and the fastest GPU on Lightning AI.**
> You get 4 free hours monthly — 100 epochs trains in ~25 min, so 1 hour covers a full run.
> Use T4 for quick debugging to save H200 hours.

---
## Cell 1 — Check GPU

In [1]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   PyTorch: {torch.__version__}")
    gpu = torch.cuda.get_device_name(0)
    if "H200" in gpu:
        print("   ✅ H200 confirmed — fastest GPU on Lightning AI, optimal for this workload")
    elif "H100" in gpu:
        print("   ✅ H100 detected — excellent performance")
    else:
        print(f"   ⚠️  Running on {gpu}. Switch to H200 for fastest training.")
else:
    raise RuntimeError(
        "❌ No GPU found.\n"
        "In Lightning AI: click the compute icon (top-right) → select H200 → wait for switch"
    )

✅ GPU: NVIDIA H200
   VRAM: 150.1 GB
   PyTorch: 2.8.0+cu128
   ✅ H200 confirmed — fastest GPU on Lightning AI, optimal for this workload


---
## Cell 2 — Clone the FSR Repository

Lightning AI working directory is `/teamspace/studios/this_studio/`.
The repo is cloned once and **stays there permanently** between sessions — this cell skips the clone if the folder already exists.

In [2]:
import os

WORK_DIR = '/teamspace/studios/this_studio'
FSR_DIR  = '/teamspace/studios/this_studio/FSR'

os.makedirs(WORK_DIR, exist_ok=True)

if os.path.exists(FSR_DIR):
    print(f'✅ FSR repo already exists at {FSR_DIR} — skipping clone')
    print('   (Storage is persistent in Lightning AI — no need to re-clone)')
else:
    os.chdir(WORK_DIR)
    os.system('git clone https://github.com/wkim97/FSR.git')
    print('✅ Repo cloned successfully')

os.chdir(FSR_DIR)
print(f'\n📂 Working directory: {os.getcwd()}')
os.system("echo '--- Repo contents ---' && ls")

Cloning into 'FSR'...

✅ Repo cloned successfully

📂 Working directory: /teamspace/studios/this_studio/FSR
--- Repo contents ---
LICENSE.txt
README.md
advertorch_fsr
attacks
environment.yml
figures
metric
models
run.sh
test.py
train.py


0

---
## Cell 3 — Install Dependencies

Lightning AI pre-installs PyTorch and torchvision. Only `tqdm` needs confirming.

In [3]:
!pip install tqdm --quiet
import tqdm, torchvision
print(f'torchvision : {torchvision.__version__}')
print(f'tqdm        : {tqdm.__version__}')
print('✅ All dependencies OK')

torchvision : 0.23.0+cu128
tqdm        : 4.67.3
✅ All dependencies OK


---
## Cell 4 — Fix Both Bugs in the Repo

Same two bugs as in the Kaggle version. Safe to re-run — checks before patching.

In [4]:
import os, subprocess, sys

os.chdir('/teamspace/studios/this_studio/FSR')

# ── Bug 1: test.py — torch.load parenthesis in wrong place ──────────────────
with open('test.py', 'r') as f:
    src = f.read()

BUGGY = ("torch.load('./weights/{}/{}/{}.pth'"
         ".format(args.dataset, args.model, args.load_name, map_location=device))")
FIXED = ("torch.load('./weights/{}/{}/{}.pth'"
         ".format(args.dataset, args.model, args.load_name), map_location=device)")

if BUGGY in src:
    src = src.replace(BUGGY, FIXED)
    with open('test.py', 'w') as f:
        f.write(src)
    print('✅ Bug 1 fixed — test.py torch.load parenthesis')
elif FIXED in src:
    print('✅ Bug 1 already correct')
else:
    print('⚠️  Bug 1 pattern not found — inspect test.py manually')


# ── Bug 2: models/BaseModel.py — abstractclassmethod removed in Python 3.11 ─
with open('models/BaseModel.py', 'r') as f:
    src = f.read()

src = src.replace(
    'from abc import ABC, abstractclassmethod',
    'from abc import ABC, abstractmethod'
)
src = src.replace('@abstractclassmethod', '@abstractmethod')

with open('models/BaseModel.py', 'w') as f:
    f.write(src)
print('✅ Bug 2 fixed — models/BaseModel.py abstractclassmethod → abstractmethod')


# ── Verify ───────────────────────────────────────────────────────────────────
r1 = subprocess.run(
    [sys.executable, '-c',
     'import sys; sys.path.insert(0,"."); from models.BaseModel import BaseModelDNN; print("BaseModel OK")'],
    capture_output=True, text=True, cwd='/teamspace/studios/this_studio/FSR'
)
print(r1.stdout.strip() or r1.stderr.strip())

print('\n--- Patched test.py torch.load line ---')
for line in open('test.py'):
    if 'torch.load' in line:
        print(line.rstrip())

✅ Bug 1 fixed — test.py torch.load parenthesis
✅ Bug 2 fixed — models/BaseModel.py abstractclassmethod → abstractmethod
BaseModel OK

--- Patched test.py torch.load line ---
    checkpoint = torch.load('./weights/{}/{}/{}.pth'.format(args.dataset, args.model, args.load_name), map_location=device)


---
## Cell 5 — Checkpoint Resume Patch

Run this cell **once after cloning**. It patches `train.py` so that:
- A full resume checkpoint (`model + optimizer + epoch number`) is saved after **every epoch**
- On restart, training **automatically continues** from the last completed epoch
- The final `.pth` weights file is still written as normal

> Because Lightning AI storage is **persistent**, the checkpoint file
> survives session end. Just reopen the Studio and run Cell 7 again.

In [5]:
import os, re

os.chdir('/teamspace/studios/this_studio/FSR')

with open('train.py', 'r') as f:
    src = f.read()

MARKER = '# <<RESUME_PATCH_APPLIED>>'
if MARKER in src:
    print('✅ Resume patch already applied — skipping')
else:
    # 1. Add --resume_ckpt argument (no leading indent — top-level argparse)
    RESUME_ARG = (
        "parser.add_argument('--resume_ckpt', type=str, default='',\n"
        "                    help='path to checkpoint to resume from (auto-detected if empty)')\n"
    )
    src = src.replace(
        'args = parser.parse_args()',
        RESUME_ARG + 'args = parser.parse_args()'
    )

    # 2. Resume block inserted before the training loop
    RESUME_BLOCK = (
        '# ── Resume from checkpoint if available ──────────────────────────────\n'
        '_ckpt_path = args.resume_ckpt or os.path.join(\n'
        "    './weights', args.dataset, args.model,\n"
        "    args.save_name + '_ckpt.pth')\n"
        'start_epoch = 1\n'
        'if os.path.exists(_ckpt_path):\n'
        "    print(f'[Resume] Loading checkpoint: {_ckpt_path}')\n"
        "    _ckpt = torch.load(_ckpt_path, map_location=device)\n"
        "    net.load_state_dict(_ckpt['model'])\n"
        "    optimizer.load_state_dict(_ckpt['optimizer'])\n"
        "    start_epoch = _ckpt['epoch'] + 1\n"
        "    print(f'[Resume] ✅ Resuming from epoch {start_epoch} / {args.epoch}')\n"
        'else:\n'
        "    print('[Resume] No checkpoint found — starting fresh from epoch 1')\n"
        '\n'
    )

    loop_match = re.search(r'^for epoch in range\(', src, re.MULTILINE)
    if not loop_match:
        print('⚠️  Could not find training loop — inspect train.py manually')
    else:
        ins = loop_match.start()
        src = src[:ins] + RESUME_BLOCK + src[ins:]

        # 3. Replace the range() call to use start_epoch
        for old_range in [
            'for epoch in range(start_epoch, args.epoch + 1)',
            'for epoch in range(1, args.epoch + 1)',
            'for epoch in range(args.epoch)',
        ]:
            if old_range in src:
                src = src.replace(old_range,
                    'for epoch in range(start_epoch, args.epoch + 1)')
                break

        # 4. Per-epoch checkpoint save block
        EPOCH_SAVE = (
            '    # ── Per-epoch checkpoint (enables resume on Lightning AI) ──────\n'
            "    _ckpt_dir = os.path.join('./weights', args.dataset, args.model)\n"
            '    os.makedirs(_ckpt_dir, exist_ok=True)\n'
            "    _ckpt_save_path = os.path.join(_ckpt_dir, args.save_name + '_ckpt.pth')\n"
            '    torch.save({\n'
            "        'epoch'    : epoch,\n"
            "        'model'    : net.state_dict(),\n"
            "        'optimizer': optimizer.state_dict(),\n"
            '    }, _ckpt_save_path)\n'
            "    print(f'[Checkpoint] ✅ Epoch {epoch}/{args.epoch} saved → {_ckpt_save_path}')\n"
            '    # ───────────────────────────────────────────────────────────────\n'
        )

        top_save = re.search(r'^torch\.save\(', src, re.MULTILINE)
        if top_save:
            src = src[:top_save.start()] + EPOCH_SAVE + '\n' + src[top_save.start():]
        else:
            m = re.search(r'^    torch\.save\(', src, re.MULTILINE)
            if m:
                src = src[:m.start()] + EPOCH_SAVE + src[m.start():]
            else:
                print('⚠️  torch.save not found — epoch checkpointing not injected')

        src = MARKER + '\n' + src

        with open('train.py', 'w') as f:
            f.write(src)
        print('✅ Resume patch applied to train.py')
        print('   • Per-epoch checkpoint → <save_name>_ckpt.pth')
        print('   • Auto-resumes on restart (Lightning AI storage is persistent)')

    print('\n--- Training loop header (sanity check) ---')
    for line in open('train.py'):
        if 'for epoch in range' in line:
            print(line.rstrip())
            break

✅ Resume patch applied to train.py
   • Per-epoch checkpoint → <save_name>_ckpt.pth
   • Auto-resumes on restart (Lightning AI storage is persistent)

--- Training loop header (sanity check) ---
for epoch in range(start_epoch, args.epoch + 1):


In [6]:
# --- Cell 5.5: Inject Learning Rate Scheduler Patch ---
import os

os.chdir('/teamspace/studios/this_studio/FSR')

with open('train.py', 'r') as f:
    lines = f.readlines()

# Check if we already patched it to prevent double-patching
if any('MultiStepLR' in line for line in lines):
    print("✅ LR Scheduler patch already applied.")
else:
    new_lines = []
    for line in lines:
        # 1. Inject scheduler.step() right BEFORE the checkpoint save block
        if '# ── Per-epoch checkpoint' in line:
            indent = line[:len(line) - len(line.lstrip())] # Keep correct indentation
            new_lines.append(indent + "scheduler.step()\n")
        
        new_lines.append(line)
        
        # 2. Inject scheduler initialization right AFTER the optimizer is defined
        if 'optimizer = optim.SGD' in line:
            indent = line[:len(line) - len(line.lstrip())] # Keep correct indentation
            new_lines.append(indent + "scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[75, 90], gamma=0.1)\n")

    # Save the patched file
    with open('train.py', 'w') as f:
        f.writelines(new_lines)
        
    print("✅ train.py successfully patched:")
    print("   • Added MultiStepLR scheduler (milestones=[75, 90])")
    print("   • Added scheduler.step() to the end of the epoch loop")

✅ train.py successfully patched:
   • Added MultiStepLR scheduler (milestones=[75, 90])
   • Added scheduler.step() to the end of the epoch loop


---
## Cell 6 — Configuration

Edit these values to control your experiment.

| Setting | Quick demo | Full paper results |
|---------|-----------|-------------------|
| `NUM_EPOCHS` | 5 (~1 min on H200) | 100 (~25 min on H200) |

> ⚡ **H200 is ~5–6× faster than T4.** 100 epochs takes ~25 min.
> You have 4 free hours monthly — enough for **9 full runs** on CIFAR-10.
> Switch to T4 for quick debugging to save H200 hours.

In [15]:
# ─── YOUR SETTINGS ───────────────────────────────────────────────────────────

SAVE_NAME  = 'cifar10_resnet18'   # name for saved checkpoint file
DATASET    = 'cifar10'            # 'cifar10'  |  'svhn'
MODEL      = 'resnet18'           # 'resnet18' | 'vgg16' | 'wideresnet34'
DEVICE     = 0                    # GPU index (keep as 0)

NUM_EPOCHS = 100    # set to 5 for a quick demo, 100 for full paper results
BATCH_SIZE = 128   # H200 has 80 GB VRAM — 512 maximises GPU utilisation

# Adversarial training hyperparameters (paper defaults for CIFAR-10)
LR      = 0.1   # use 0.01 for SVHN
EPS     = 8.0    # perturbation budget in units of /255
ALPHA   = 0.25   # PGD step size = ALPHA * EPS/255  |  use 0.125 for SVHN
TAU     = 0.1    # Gumbel-softmax temperature
LAM_SEP = 1.0    # weight on separation loss L_sep
LAM_REC = 1.0    # weight on recalibration loss L_rec

# ─────────────────────────────────────────────────────────────────────────────
print(f'Model      : {MODEL}')
print(f'Dataset    : {DATASET}')
print(f'Epochs     : {NUM_EPOCHS}')
print(f'Batch size : {BATCH_SIZE}  ← increased to 512 for H200 (80 GB VRAM)')
print(f'LR={LR} | eps={EPS}/255 | alpha={ALPHA} | tau={TAU}')
print(f'lam_sep={LAM_SEP} | lam_rec={LAM_REC}')

Model      : resnet18
Dataset    : cifar10
Epochs     : 100
Batch size : 128  ← increased to 512 for H200 (80 GB VRAM)
LR=0.1 | eps=8.0/255 | alpha=0.25 | tau=0.1
lam_sep=1.0 | lam_rec=1.0


---
## Cell 7 — Train

CIFAR-10 (~170 MB) downloads automatically on the first run.

Weights are saved to `weights/` inside the FSR folder after **every epoch**.
Lightning AI storage is **persistent** — if your session ends, just reopen
the Studio and run this cell again. Training will **auto-resume** from the
last completed epoch.

**Expected output per epoch on H200 (~15 sec/epoch):**
```
Epoch: 1
cifar10_resnet18 (Train) Epoch: 1/100: 100%|...| Adv_Acc=10.7%
cifar10_resnet18 (Test)  Epoch: 1/100: 100%|...| Adv_Acc=10.5%
[Checkpoint] ✅ Epoch 1/100 saved → ./weights/cifar10/resnet18/cifar10_resnet18_ckpt.pth
```

⚡ **100 epochs ≈ 20–25 min on H200** (vs ~45 min on L40S, ~3 hrs on T4)

In [20]:
import os
import shutil

# 1. Force delete old weights so it actually trains
weights_dir = '/teamspace/studios/this_studio/FSR/weights/cifar10/resnet18'
if os.path.exists(weights_dir):
    shutil.rmtree(weights_dir)
    print("🗑️ Old weights deleted. The model will now train from scratch.")
else:
    print("🗑️ No old weights found.")

# 2. Patch fsr_visualize.py to evaluate the whole dataset, not just 10 batches
viz_path = '/teamspace/studios/this_studio/FSR/fsr_visualize.py'
if os.path.exists(viz_path):
    with open(viz_path, 'r') as f:
        viz_code = f.read()
    
    # Replace the speed limit with a number larger than the dataset
    viz_code = viz_code.replace("max_batches=10", "max_batches=9999")
    
    with open(viz_path, 'w') as f:
        f.write(viz_code)
    print("✅ Patched fsr_visualize.py to evaluate the FULL test set.")

# 3. Double-check train.py has the LR scheduler
train_path = '/teamspace/studios/this_studio/FSR/train.py'
with open(train_path, 'r') as f:
    train_code = f.read()

if 'MultiStepLR' not in train_code:
    new_lines = []
    with open(train_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        if '# ── Per-epoch checkpoint' in line:
            indent = line[:len(line) - len(line.lstrip())]
            new_lines.append(indent + "scheduler.step()\n")
        
        new_lines.append(line)
        
        if 'optimizer = optim.SGD' in line:
            indent = line[:len(line) - len(line.lstrip())]
            new_lines.append(indent + "scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[75, 90], gamma=0.1)\n")
            
    with open(train_path, 'w') as f:
        f.writelines(new_lines)
    print("✅ Learning Rate scheduler added to train.py.")
else:
    print("✅ train.py already has the Learning Rate scheduler.")

🗑️ Old weights deleted. The model will now train from scratch.
✅ train.py already has the Learning Rate scheduler.


In [21]:
import os
os.chdir('/teamspace/studios/this_studio/FSR')

!python train.py \
    --save_name {SAVE_NAME} \
    --dataset   {DATASET}   \
    --model     {MODEL}     \
    --device    {DEVICE}    \
    --epoch     {NUM_EPOCHS} \
    --bs        {BATCH_SIZE} \
    --lr        {LR}        \
    --eps       {EPS}       \
    --alpha     {ALPHA}     \
    --tau       {TAU}       \
    --lam_sep   {LAM_SEP}   \
    --lam_rec   {LAM_REC}

[Resume] No checkpoint found — starting fresh from epoch 1

Epoch: 1
cifar10_resnet18 (Train) Epoch: 1/100: : 50000it [00:35, 1405.37it/s, Adv_Acc=10.082%, Adv_Loss=2.357, Rec_Loss=2.641, Sep_Loss=4.337]
cifar10_resnet18 (Test) Epoch: 1/100: : 10000it [00:05, 1953.06it/s, Adv_Acc=9.420%, Adv_Loss=2.307, Ori_Acc=11.780%, Ori_Loss=2.297]
[Checkpoint] ✅ Epoch 1/100 saved → ./weights/cifar10/resnet18/cifar10_resnet18_ckpt.pth

Epoch: 2
cifar10_resnet18 (Train) Epoch: 2/100: : 50000it [00:34, 1436.47it/s, Adv_Acc=10.560%, Adv_Loss=2.305, Rec_Loss=2.572, Sep_Loss=4.131]
cifar10_resnet18 (Test) Epoch: 2/100: : 10000it [00:05, 1985.90it/s, Adv_Acc=13.720%, Adv_Loss=2.290, Ori_Acc=16.300%, Ori_Loss=2.224]
[Checkpoint] ✅ Epoch 2/100 saved → ./weights/cifar10/resnet18/cifar10_resnet18_ckpt.pth

Epoch: 3
cifar10_resnet18 (Train) Epoch: 3/100: : 50000it [00:34, 1435.60it/s, Adv_Acc=16.866%, Adv_Loss=2.202, Rec_Loss=2.377, Sep_Loss=3.524]
cifar10_resnet18 (Test) Epoch: 3/100: : 10000it [00:05, 1988.

---
## Cell 8 — Evaluate Robustness

Tests the saved checkpoint against FGSM, PGD-20, PGD-100, C&W, and Ensemble.

In [23]:
import os

filepath = '/teamspace/studios/this_studio/FSR/test.py'

with open(filepath, 'r') as f:
    content = f.read()

# Replace the hardcoded 5000 limit with 10000 (full CIFAR-10 test set)
if '5000' in content:
    content = content.replace('5000', '10000')
    
    with open(filepath, 'w') as f:
        f.write(content)
    print("✅ test.py successfully patched! It will now evaluate on all 10,000 images.")
else:
    print("⚠️ Could not find the '5000' limit in test.py. It might already be patched.")

⚠️ Could not find the '5000' limit in test.py. It might already be patched.


In [24]:
import os
os.chdir('/teamspace/studios/this_studio/FSR')

!python test.py \
    --load_name {SAVE_NAME} \
    --dataset   {DATASET}   \
    --model     {MODEL}     \
    --device    {DEVICE}    \
    --tau       {TAU}       \
    --bs        {BATCH_SIZE}

FGSM: Ori Acc: 82.86%	Adv Acc: 57.71%
PGD-20: Ori Acc: 82.86%	Adv Acc: 52.38%
PGD-100: Ori Acc: 82.86%	Adv Acc: 50.81%
C&W: Ori Acc: 82.86%	Adv Acc: 50.13%
Ensemble : 48.42%


---
## Cell 9 — Find Your Saved Files

Lightning AI stores everything persistently. This cell shows exactly where
your weights and checkpoints are saved and how to download them.

In [25]:
import os, glob

weights_dir = f'/teamspace/studios/this_studio/FSR/weights/{DATASET}/{MODEL}'
print(f'📂 Weights directory: {weights_dir}')
print()

if os.path.exists(weights_dir):
    files = glob.glob(weights_dir + '/*.pth')
    if files:
        for f in sorted(files):
            size_mb = os.path.getsize(f) / 1e6
            print(f'  ✅ {os.path.basename(f)}  ({size_mb:.1f} MB)')
    else:
        print('  ⚠️  No .pth files yet — run Cell 7 (Train) first')
else:
    print('  ⚠️  Directory does not exist yet — run Cell 7 (Train) first')

print()
print('💡 To download files to your local machine:')
print('   Lightning AI → File Browser icon (left sidebar)')
print('   → navigate to the weights folder → right-click file → Download')

📂 Weights directory: /teamspace/studios/this_studio/FSR/weights/cifar10/resnet18

  ✅ cifar10_resnet18.pth  (49.8 MB)
  ✅ cifar10_resnet18_ckpt.pth  (99.6 MB)

💡 To download files to your local machine:
   Lightning AI → File Browser icon (left sidebar)
   → navigate to the weights folder → right-click file → Download


---
## Cell 10 — Other Model / Dataset Combinations (Optional)

Uncomment ONE block at a time.

| Model | Dataset | Est. time on H200 |
|-------|---------|-------------------|
| ResNet-18 | CIFAR-10 | ~20–25 min |
| VGG16 | CIFAR-10 | ~28–35 min |
| WideResNet-34 | CIFAR-10 | ~50–60 min |
| ResNet-18 | SVHN | ~18–22 min |

In [19]:
import os
os.chdir('/teamspace/studios/this_studio/FSR')

# ── VGG16 on CIFAR-10 ──────────────────────────────────────────────────────
# !python train.py --save_name cifar10_vgg16 --dataset cifar10 --model vgg16 --device 0
# !python test.py  --load_name cifar10_vgg16 --dataset cifar10 --model vgg16 --device 0

# ── WideResNet-34-10 on CIFAR-10 ───────────────────────────────────────────
# !python train.py --save_name cifar10_wideresnet34 --dataset cifar10 --model wideresnet34 --device 0
# !python test.py  --load_name cifar10_wideresnet34 --dataset cifar10 --model wideresnet34 --device 0

# ── ResNet-18 on SVHN (different lr and alpha!) ─────────────────────────────
# !python train.py --save_name svhn_resnet18 --dataset svhn --model resnet18 --device 0 --lr 0.01 --alpha 0.125
# !python test.py  --load_name svhn_resnet18 --dataset svhn --model resnet18 --device 0

print('Uncomment a block above and run this cell.')

Uncomment a block above and run this cell.


---
## Cell 11 — Generate PPT Images & Variables

Run this **after training is complete**.
Outputs are saved to `fsr_outputs/` — persistent on Lightning AI.

In [28]:
import os
print(os.path.exists('/teamspace/studios/this_studio/FSR/fsr_visualize.py'))

True


In [32]:
import os
import models

VIZ_PATH = '/teamspace/studios/this_studio/FSR/fsr_visualize.py'
os.chdir('/teamspace/studios/this_studio/FSR')

if not os.path.exists(VIZ_PATH):
    print('❌ fsr_visualize.py not found at', VIZ_PATH)
    print('   Upload it via Lightning AI File Browser (left sidebar)')
    print('   → drag fsr_visualize.py into the FSR folder')
else:
    %run fsr_visualize.py
    print('\n✅ All images saved to /teamspace/studios/this_studio/FSR/fsr_outputs/')
    print('   File Browser (left sidebar) → fsr_outputs → right-click file → Download')

▶  Device : cuda:0
▶  Model  : resnet18 | Dataset: cifar10 | Save name: cifar10_resnet18
▶  Output : /teamspace/studios/this_studio/FSR/fsr_outputs

✅ Loading weights from: /teamspace/studios/this_studio/FSR/weights/cifar10/resnet18/cifar10_resnet18.pth
✅ Weights loaded (epoch: final)

📊 Evaluating accuracies
  Clean  : 83.26%
  FGSM   : 57.26%
  PGD-20 : 50.54%
  PGD-50 : 49.84%

🎨 Generating robustness bar chart...
  ✅ Saved: /teamspace/studios/this_studio/FSR/fsr_outputs/robustness_bar.png

🎨 Generating attention maps...
  ✅ Saved: /teamspace/studios/this_studio/FSR/fsr_outputs/attention_maps.png

🎨 Generating feature norm distribution...
  ✅ Saved: /teamspace/studios/this_studio/FSR/fsr_outputs/feature_norms.png

📝 Writing results summary...

═══════════════════════════════════════════════════════
✅  FSR Visualization complete!
   Output folder: /teamspace/studios/this_studio/FSR/fsr_outputs

   Files generated:
   📊 robustness_bar.png   — accuracy bar chart
   🗺️  attention_maps.p

In [33]:
!touch /teamspace/studios/this_studio/FSR/models/__init__.py

In [34]:
import shutil
import os

# Define the paths
folder_to_zip = '/teamspace/studios/this_studio/FSR'
output_filename = '/teamspace/studios/this_studio/FSR_backup' # The .zip extension is added automatically

print(f"📦 Zipping the folder: {folder_to_zip} ...")
print("⏳ This might take a minute depending on the size of the saved weights...")

# Create the zip archive
shutil.make_archive(output_filename, 'zip', folder_to_zip)

print(f"✅ Successfully created: {output_filename}.zip")
print("⬇️  You can now download it from the File Browser on the left sidebar.")

📦 Zipping the folder: /teamspace/studios/this_studio/FSR ...
⏳ This might take a minute depending on the size of the saved weights...
✅ Successfully created: /teamspace/studios/this_studio/FSR_backup.zip
⬇️  You can now download it from the File Browser on the left sidebar.
